# Figura 2 · La distribución de los salarios en España, 2008-2024

Este cuaderno crea la Figura 2 del trabajo con **Plotly**. El resultado es un archivo `figura2.html` que se coloca junto a `index.html` y se incrusta en la web con un iframe, igual que las figuras 4, 5, 7 y 8.

**Qué muestra.** Para cada año, el salario bruto anual por trabajador en cinco puntos de la distribución (percentil 10, primer cuartil, mediana, tercer cuartil y percentil 90) y la media. Las franjas reparten a los asalariados en grupos, y en el centro de cada una se indica qué porcentaje de ellos reúne. Al pasar el ratón por un año se ven los euros de cada punto.

**Datos.** La tabla `data/processed/salarios_ccaa.csv`, que genera el cuaderno principal (`TP_08MBID_preparacion_datos.ipynb`, paso 1). No se crea ninguna tabla nueva.

**Fuente.** INE, Encuesta Anual de Estructura Salarial (tabla 28191). Ámbito: total nacional, ambos sexos, todas las jornadas.

## 1. Librerías y estilo

Los mismos colores que en la web y en la Figura 1: azul para el salario medio y naranja para el mediano.

In [1]:
import pandas as pd
import plotly.graph_objects as go

AZUL, NARANJA, TINTA, TINTA2, GRIS = "#2A78D6", "#EB6834", "#14130F", "#55534C", "#D9D6CC"
BANDA_CLARA, BANDA_OSCURA = "#E4E1D8", "#C9C5B8"   # P10-Q1 y Q3-P90 / Q1-Q3

def euros(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

## 2. Datos

Leemos la tabla del repositorio público y nos quedamos con la fila **Nacional**. Los valores son euros brutos anuales por trabajador.

In [2]:
URL = ("https://raw.githubusercontent.com/sortaria24-max/08MBID_TrabajoPractico/"
       "main/data/processed/salarios_ccaa.csv")
s = pd.read_csv(URL)
nac = s[s.ccaa == "Nacional"].sort_values("anio").set_index("anio")
nac = nac[["p10", "q1", "mediana", "media", "q3", "p90"]]
nac.round(0).astype(int)

,p10,q1,mediana,media,q3,p90
anio,,,,,,
2008,8904,13343,18244,21883,26874,38558
2009,8723,13587,18803,22511,28025,40124
2010,8644,13603,19017,22790,28255,40811
2011,8402,13554,19304,22899,28659,40681
2012,7979,13369,19041,22726,28396,40808
2013,7692,13039,19030,22698,28564,41108
2014,7626,13218,19264,22858,28783,41350
2015,7962,13415,19466,23106,29164,41649
2016,8095,13370,19433,23156,29192,41855


## 3. Comprobaciones

Antes de dibujar comprobamos que los percentiles están ordenados en todos los años y que la media supera siempre a la mediana.

In [3]:
orden = ["p10", "q1", "mediana", "q3", "p90"]
assert (nac[orden].diff(axis=1).iloc[:, 1:] > 0).all().all(), "Percentiles desordenados"
assert (nac.media > nac.mediana).all(), "La media debería superar a la mediana"
fin = nac.loc[2024]
print("2024:", ", ".join(f"{c} {euros(fin[c])}" for c in nac.columns))

2024: p10 12.018 €, q1 17.457 €, mediana 24.497 €, media 29.540 €, q3 36.970 €, p90 52.515 €


## 4. Gráfico

- **Franjas:** las claras van del percentil 10 al primer cuartil y del tercer cuartil al percentil 90 (un 15 % de los asalariados cada una); la oscura, del primer al tercer cuartil (el 50 %, que la mediana parte en dos mitades del 25 %). Por debajo del percentil 10 y por encima del 90 queda un 10 % en cada extremo. El porcentaje va escrito en el centro de cada franja.
- **Líneas:** la mediana en naranja y la media en azul.
- **Etiquetas directas** a la derecha, con el valor de 2024, para no necesitar leyenda.
- **Interacción:** al pasar el ratón por un año, un recuadro muestra los seis valores de ese año, ordenados de mayor a menor.
- El eje vertical empieza en 0 porque son euros: así la altura de cada franja es proporcional a la diferencia real de sueldos.

In [4]:
x = nac.index
fig = go.Figure()

# Franjas (sin recuadro al pasar el ratón: los valores salen en las líneas de abajo)
fig.add_trace(go.Scatter(x=x, y=nac.p10, mode="lines", line=dict(width=0), hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=x, y=nac.q1, mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor=BANDA_CLARA, hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=x, y=nac.q3, mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor=BANDA_OSCURA, hoverinfo="skip", showlegend=False))
fig.add_trace(go.Scatter(x=x, y=nac.p90, mode="lines", line=dict(width=0), fill="tonexty",
                         fillcolor=BANDA_CLARA, hoverinfo="skip", showlegend=False))

# Valores al pasar el ratón. Plotly los lista en orden inverso al de las capas,
# así que se añaden de menor a mayor para que el recuadro salga de mayor a menor.
series = [("p90", "Percentil 90", TINTA2, 0), ("q3", "Tercer cuartil", TINTA2, 0),
          ("media", "Salario medio", AZUL, 2.6), ("mediana", "Salario mediano", NARANJA, 2.6),
          ("q1", "Primer cuartil", TINTA2, 0), ("p10", "Percentil 10", TINTA2, 0)]
for col, nombre, color, grosor in reversed(series):
    fig.add_trace(go.Scatter(
        x=x, y=nac[col], name=nombre, mode="lines", showlegend=False,
        line=dict(color=color, width=grosor if grosor else 0.01),
        hovertemplate=f"{nombre}: %{{y:,.0f}} €<extra></extra>"))

# Etiquetas directas a la derecha (2024)
anot = []
desplaza = {"media": 1300, "mediana": -1300}
for col, nombre, color, grosor in series:
    v = nac.loc[2024, col]
    anot.append(dict(x=2024, y=v + desplaza.get(col, 0), xanchor="left", xshift=8, showarrow=False,
                     text=(f"<b>{nombre}  {euros(v)}</b>" if grosor else f"{nombre}  {euros(v)}"),
                     font=dict(size=12, color=color)))

# Porcentaje de asalariados en cada franja, escrito en su centro (año 2012)
f = nac.loc[2012]
rotulos = [(f.p10 / 2, "10 %", TINTA2), ((f.p10 + f.q1) / 2, "15 %", TINTA),
           ((f.q1 + f.mediana) / 2, "25 %", TINTA), ((f.media + f.q3) / 2, "25 %", TINTA),
           ((f.q3 + f.p90) / 2, "15 %", TINTA), (f.p90 + 4000, "10 %", TINTA2)]
for y, texto, color in rotulos:
    anot.append(dict(x=2012.5, y=y, text=f"<b>{texto}</b>", showarrow=False, font=dict(size=13, color=color)))

fig.update_layout(
    annotations=anot,
    xaxis=dict(range=[2008, 2024], dtick=2, showgrid=False, zeroline=False, linecolor=TINTA2,
               fixedrange=True, hoverformat="d"),
    yaxis=dict(range=[0, 56000], dtick=10000, tickformat=",.0f", ticksuffix=" €", gridcolor="#E6E3DA",
               zeroline=False, fixedrange=True),
    hovermode="x unified", hoverlabel=dict(font_size=13, bgcolor="white", bordercolor=GRIS),
    font=dict(family="Inter, system-ui, -apple-system, Segoe UI, sans-serif", color=TINTA2, size=12),
    paper_bgcolor="white", plot_bgcolor="white", height=560, separators=",.",
    margin=dict(l=70, r=200, t=20, b=40),
)
fig.show(config={"displaylogo": False})

## 5. Guardar

- `figura2.html`: el gráfico interactivo, como página propia. Se coloca en StackBlitz junto a `index.html`, que lo incrusta con un iframe.
- Para el PDF: el icono de la cámara, arriba a la derecha del gráfico, descarga `figura2.png`. Súbelo a la carpeta `imagenes` del repositorio.

In [5]:
config = {"displaylogo": False, "responsive": True,
          "toImageButtonOptions": {"format": "png", "filename": "figura2", "scale": 2, "width": 1000, "height": 560}}
fig.write_html("figura2.html", include_plotlyjs="cdn", full_html=True, config=config)

# Misma tipografía que la web
html = open("figura2.html", encoding="utf-8").read()
fuente = ('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&display=swap">'
          '<style>body{margin:0}</style>')
open("figura2.html", "w", encoding="utf-8").write(html.replace("<head>", "<head>" + fuente, 1))
print("Guardado: figura2.html")

try:
    from google.colab import files
    files.download("figura2.html")
except ImportError:
    pass

Guardado: figura2.html
